# Module 2: Deploy + Govern

> Part of the **Modular Workshops** series. Standalone, ~25 min.

Moving the agent to production requires two things:

- **A governance layer** — workspace-level policies that govern every model call (PII and secrets redaction, allow-lists, cost caps). The LangSmith **LLM Gateway** enforces them.
- **Production-ready deployment infrastructure** — managed runtime for the agent. **Managed Deep Agents (MDA)** ships the Deep Agents harness from Module 1 straight to LangSmith's Agent Server — 30+ endpoints, persistence, HITL, and Studio out of the box, no backend to build.

This module wires up both — policies first, then ships the policy-aware agent.

## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import os

print("LANGSMITH_API_KEY set:        ", bool(os.environ.get("LANGSMITH_API_KEY")))
print("LANGSMITH_API_KEY_GATEWAY set:", bool(os.environ.get("LANGSMITH_API_KEY_GATEWAY")))
print("WORKSPACE_ID set:             ", bool(os.environ.get("WORKSPACE_ID")))
print("OPENAI_API_KEY set:           ", bool(os.environ.get("OPENAI_API_KEY")))
print("TAVILY_API_KEY set:           ", bool(os.environ.get("TAVILY_API_KEY")))

---
# Part 1. Govern — Workspace Policies via the LangSmith Gateway

> The LangSmith LLM Gateway needs to be enabled in your LangSmith org. If it's not visible in your workspace, it needs to be enabled at the organization level, which can be done by your LangSmith administrator. Gateway is currently available on cloud; self-hosted LangSmith support is coming.

The **LangSmith LLM Gateway** is a managed proxy that sits between any client and the model provider (OpenAI, Anthropic, Bedrock, …). Point a model client at the gateway's `base_url` and every request flows through it — no SDK changes needed.

**Gateway policies** are the controls applied at the gateway. They're declared once at the workspace level and enforced on every call, regardless of which agent or notebook is making it.

We'll wire up a policy that redacts PII and secrets, prove it catches a bad prompt, then promote the gateway from a smoke test to the default model config — so both this notebook and the agent we're about to deploy inherit it.

### 1.1 Create a PII / secrets redaction policy

The gateway policy is a workspace-level resource — one POST to `/v1/platform/gateway-policies` and every model call made by every member of the workspace is subject to it.

The payload below declares:

- `action=block` — intercept matched content before forwarding upstream
- `subject_matchers` — apply to this workspace's traffic
- `config.detect.pii / .secrets` — what to look for

In [ ]:
import requests

BASE_URL = "https://api.smith.langchain.com"
api_key = os.environ["LANGSMITH_API_KEY"]
workspace_id = os.environ["WORKSPACE_ID"]

policy = {
    "name": "redact-pii-secrets",
    "description": "Workspace-level PII and secrets redaction",
    "policy_type": "guard",
    "action": "block",
    "subject_matchers": [{"key": "workspace_id", "value": workspace_id}],
    "config": {"version": 1, "detect": {"pii": True, "secrets": True}},
    "enabled": True,
}

resp = requests.post(
    f"{BASE_URL}/v1/platform/gateway-policies",
    headers={"X-Api-Key": api_key, "Content-Type": "application/json"},
    json=policy,
    timeout=30,
)
resp.raise_for_status()
created = resp.json()
print("Policy created.")
print("  id:     ", created.get("id"))
print("  name:   ", created.get("name"))
print("  action: ", created.get("action"))
print("  enabled:", created.get("enabled"))

### 1.2 Route a minimal agent through the gateway

The policy is live workspace-wide, but it only applies to traffic that *goes through the gateway*. To opt a client in, swap its `base_url` to `https://gateway.smith.langchain.com/<provider>`. No other code changes.

Below: a bare `create_deep_agent` whose model client is gateway-routed. We'll exercise it with a clean prompt first to confirm normal traffic still flows.

In [ ]:
from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model
from utils.utils import clear_local_env_vars

# Ensure this local demo uses the workshop's gateway key, not any company-gateway overrides.
clear_local_env_vars()

gateway_model = init_chat_model(
    model="gpt-5.6-terra",
    model_provider="openai",
    base_url="https://gateway.smith.langchain.com/openai",
    use_responses_api=True,
    api_key=os.environ["LANGSMITH_API_KEY_GATEWAY"],
)

demo_agent = create_deep_agent(
    model=gateway_model,
    system_prompt=(
        "You are a helpful assistant behind the LangSmith Gateway, which redacts "
        "sensitive data before you see it, replacing each value with a token like "
        "[SAFE_TO_USE:EMAIL_ADDRESS_a1b2c3]. Treat each token as the real value, "
        "answer normally, and repeat tokens verbatim — the Gateway restores the real "
        "value in your reply. When a reply involves redacted data, start with one line: "
        "🔒 Gateway redacted N sensitive value(s) before I saw them."
    ),
)

ok = demo_agent.invoke({
    "messages": [{"role": "user", "content": "In one sentence, what does LangSmith do?"}]
})
print(ok["messages"][-1].text)

### 1.3 Watch the policy in action

Same agent, same gateway, but the prompt below carries a (fake) email. The gateway inspects the request, matches the `guard` policy, and intercepts the PII before forwarding upstream.

The text response won't reveal whether enforcement happened — a clean response comes back either way. To see the effect, open this run's trace in LangSmith: the request payload the gateway sent upstream will show the PII removed.

In [ ]:
pii_prompt = (
    "Contact me at jane.doe@example.com for the quarterly report."
)

result = demo_agent.invoke({
    "messages": [{"role": "user", "content": pii_prompt}]
})
print(result["messages"][-1].text)

### 1.4 Make the deployable agent gateway-aware

The agent we're about to deploy lives in its own project at `agents/deep_agent/` — a self-contained [Managed Deep Agents](https://docs.langchain.com/langsmith/python/managed-deep-agents-overview) (MDA) project with its own `pyproject.toml` and `.env`, so the `mda` toolchain stays isolated from this notebook's environment. Its model config lives directly in `agents/deep_agent/agent.py`, not in `utils/models.py`.

Before (the current default):
```python
# agents/deep_agent/agent.py
model = "openai:gpt-5.6-luna"
```

After (through the LangSmith Gateway):
```python
# agents/deep_agent/agent.py
model = init_chat_model(
    "openai:gpt-5.6-luna",
    base_url="https://gateway.smith.langchain.com/openai",
    use_responses_api=True,
    api_key=os.environ["LANGSMITH_API_KEY_GATEWAY"],
)
```

To switch, comment out the direct line and uncomment the gateway block in that file, then add `LANGSMITH_API_KEY_GATEWAY` to `agents/deep_agent/.env` (same value as `LANGSMITH_API_KEY`).

> **Why `LANGSMITH_API_KEY_GATEWAY` and not `LANGSMITH_API_KEY`?**
> `LANGSMITH_API_KEY` is reserved — the platform uses it to authenticate the deployment itself, and `mda deploy` excludes reserved names from the forwarded secrets. `LANGSMITH_API_KEY_GATEWAY` is just that same key under a name the platform doesn't reserve, so the deployed agent can pass it along to the gateway.

Because `agents/deep_agent/` runs in its own environment (installed with `uv sync` in Part 2), this notebook's kernel can't import it to show the change taking effect — you'll see it once deployed, by checking the trace's request payload like in §1.3.

---
# Part 2. Deploy — Ship the (Now-Governed) Agent

Policies are in place workspace-wide. Next, ship the deployment itself.

We deploy the agent in `agents/deep_agent/` with [Managed Deep Agents](https://docs.langchain.com/langsmith/python/managed-deep-agents-overview) (MDA) — a code-first way to ship the Deep Agents harness from Module 1 without hand-building a backend, store, checkpointer, or server around it. `define_deep_agent` replaces `create_deep_agent`: it returns a *definition*, and the managed runtime builds and hosts the actual agent on LangSmith's Agent Server.

## 2.1 Project structure

A Managed Deep Agents project is a directory with an `agent.py` at its root — `agents/deep_agent/`, in this case. Unlike the notebook-built agents in Module 1, it's a self-contained project with its own `pyproject.toml` and `.env`, so its dependencies and the `mda` CLI stay isolated from this workshop's shared environment.

In [ ]:
import os

agent_dir = str(project_root / "agents" / "deep_agent")
print("agents/deep_agent/")
print("---")

for root, dirs, files in os.walk(agent_dir):
    # Skip __pycache__ and build output for clarity
    dirs[:] = [d for d in dirs if d not in ("__pycache__", ".mda")]
    level = root.replace(agent_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

## 2.2 From `create_deep_agent` to `define_deep_agent`

The agent you built across Module 1 maps onto this project as follows:

| Module 1 (`create_deep_agent`) | `agents/deep_agent/` (MDA) |
|---|---|
| `system_prompt=...`, `memory=["/AGENTS.md"]` | `instructions.md` |
| `skills=["/skills/"]` | `skills/<name>/SKILL.md` (same files) |
| Custom `StoreBackend` route for `/memories/` | `memory.py` -> `define_memory(scope="agent")`, mounted at `/memories/agent/` |
| `FilesystemBackend(virtual_mode=True)` for `/final_report.md` | No backend to declare — the managed runtime gives every run the same thread-scoped virtual filesystem by default |
| `checkpointer=MemorySaver()` | No equivalent to declare — threads and persistence come from Agent Server |
| `model`, `tools`, `subagents`, `interrupt_on` | Carry over unchanged |

`name` is new and required — a static string that becomes the agent's graph ID and the default deployment name.

**Docs:** [Managed Deep Agents overview](https://docs.langchain.com/langsmith/python/managed-deep-agents-overview) · [Move from Deep Agents](https://docs.langchain.com/langsmith/python/managed-deep-agents-migrate) · [Agent definition](https://docs.langchain.com/langsmith/python/managed-deep-agents-agent-definition)

## 2.3 agent.py + instructions.md

`agent.py` must export a module-level `agent` variable — that's what `mda build`, `mda dev`, and `mda deploy` all load. `instructions.md` holds the system prompt; it syncs to [Context Hub](https://docs.langchain.com/langsmith/python/managed-deep-agents-context-hub) on deploy, where you can edit it later without redeploying.

In [ ]:
agent_path = os.path.join(agent_dir, "agent.py")
with open(agent_path) as f:
    print("agent.py:")
    print(f.read())

instructions_path = os.path.join(agent_dir, "instructions.md")
with open(instructions_path) as f:
    print("\ninstructions.md:")
    print(f.read())

`memory.py` declares the durable memory layer mounted at `/memories/agent/` — the MDA equivalent of the `StoreBackend` route you'd otherwise wire up by hand. See §2.2 above for the full parameter mapping.

## 2.4 Local development

Three CLI commands you'll use (all run from `agents/deep_agent/`, where its own `pyproject.toml` and `agent.py` live):

```bash
# Compile the project into a managed LangGraph app, without deploying
uv run mda build

# Compile and run on the local LangGraph dev server (Studio UI + hot reload)
uv run mda dev

# Compile, sync Context Hub, upload, and deploy to LangSmith
uv run mda deploy
```

`mda dev` opens LangSmith Studio in your browser — useful to step through tool calls and approve HITL interrupts visually, same as `langgraph dev` would for a hand-built graph.

**Docs:** [Managed Deep Agents CLI reference](https://docs.langchain.com/langsmith/python/managed-deep-agents-cli) · [Local development](https://docs.langchain.com/langsmith/python/managed-deep-agents-local-development).

## 2.5 Build the project

`mda build` compiles the project into a managed LangGraph app (into `.mda/build`) — no Context Hub sync, no upload, no deploy. Use it to catch import and config errors before deploying.

In [ ]:
# `cd` and `!` run in a subshell -- chain in one line so cwd applies to the mda command.
!cd "{project_root}/agents/deep_agent" && uv sync && uv run mda build

## 2.6 Deploy to LangSmith (optional)

Run the cell below to deploy with MDA. `mda deploy` compiles the project, syncs `instructions.md` and `skills/` to Context Hub, uploads the project, and deploys it to the same [Agent Server](https://docs.langchain.com/langsmith/agent-server-overview) runtime LangSmith Deployments use. Provisioning takes a few minutes.

> The agent picks up the gateway `base_url` straight from `agents/deep_agent/agent.py` if you flipped it on in §1.4 — no extra deploy flags needed. The deployed agent is governed by the policy from §1.1 from the moment it serves its first request.

> Requires a `LANGSMITH_API_KEY` with deployment permissions — a service key (`lsv2_sk_...`), not a personal token. `mda deploy` creates a **Serverless** deployment by default, which requires a supported GCP SaaS region; use `--deployment-type dedicated` for always-on capacity elsewhere (AWS, Azure, self-hosted, hybrid).

> Everything in `agents/deep_agent/` becomes part of the build archive. `mda build` forwards non-reserved `.env` values to the deployment as secrets and excludes `.env` itself from the archive — being in `.gitignore` does not exclude it.

> **Outside the US?** `mda deploy` targets the GCP US SaaS region by default (`LANGSMITH_ENDPOINT=https://api.smith.langchain.com`). If your LangSmith org/workspace is provisioned in a different SaaS region, your API key will not authenticate against the US endpoint — set `LANGSMITH_ENDPOINT` in `agents/deep_agent/.env` to match your org before deploying: `https://eu.api.smith.langchain.com` (GCP EU), `https://apac.api.smith.langchain.com` (GCP APAC), or `https://aws.api.smith.langchain.com` (AWS US). This is about which region your workspace lives in, not where you physically run the notebook from.

Useful flags:
- `--name <name>` — deployment name (defaults to the `name` passed to `define_deep_agent`, here `deep-agent`)
- `--deployment-type dedicated` — always-on deployment; required outside GCP SaaS regions (default is `serverless`)
- `--no-wait` — trigger the build without polling for completion

In [ ]:
# Re-run this command to push a new revision; the CLI finds the existing deployment by name.
# Add `--deployment-type dedicated` for always-on capacity outside GCP SaaS regions.
!cd "{project_root}/agents/deep_agent" && uv run mda deploy

## 2.7 What you get with Managed Deep Agents

Once deployed, your agent runs on the same [Agent Server](https://docs.langchain.com/langsmith/agent-server-overview) as a hand-built LangSmith Deployment, reachable through 30+ endpoints, plus the managed layer MDA adds on top:

| Capability | What you get |
|---|---|
| **REST API** | Standard HTTP requests against `/runs`, `/threads`, `/store` |
| **Studio UI** | Visual debugger to step through state, threads, and tool calls |
| **Agent Protocol** | Stream runs and pause for human input |
| **MCP server** | Other agents can call your agent as a tool |
| **A2A** | Agent-to-agent calls with handoffs |
| **Context Hub** | Edit `instructions.md` and skills live, without a redeploy |
| **Durable memory** | `memory.py` -> `/memories/agent/`, survives restarts and threads |
| **HITL** | Interrupt and resume from any client |
| **Channels** | Wire the agent up to Slack and other messaging surfaces declaratively |
| **Schedules** | Managed cron — trigger your agent on a recurring basis |

## Recap

| Pillar | What | How |
|---|---|---|
| **Govern** | Workspace-level PII / secrets redaction | `POST /v1/platform/gateway-policies` (`action=block`) |
| **Govern** | Route model calls through the gateway | `init_chat_model(..., base_url="https://gateway.smith.langchain.com/openai")` in `agents/deep_agent/agent.py` |
| **Deploy** | Project layout | `agent.py`, `instructions.md`, `skills/`, `memory.py` in `agents/deep_agent/` |
| **Deploy** | Define the agent | `define_deep_agent(name=..., model=..., tools=..., subagents=..., interrupt_on=...)` |
| **Deploy** | Ship it | `mda deploy` |

**The production pattern:** one policy + one `base_url` change + one deploy command. The agent now lives behind a managed server, every model call passes through a governed gateway, and 30+ endpoints are available to call it.

**Next:** Module 3 — LangSmith (tracing, querying traces, offline + online evals, annotation queues).

## Clean up (optional)

Run the cell below to tear down the deployment and the LangSmith resources `mda deploy` created for it -- `mda destroy` is an alias for the same command. This only removes the hosted deployment; your local project files in `agents/deep_agent/` are untouched.

> `--yes` skips the interactive delete confirmation, which a notebook `!` cell can't answer. Drop it to confirm from a terminal instead.

**Docs:** [Delete deployments](https://docs.langchain.com/langsmith/python/managed-deep-agents-cli#delete-deployments).

In [ ]:
# Deletes the deployment named "deep-agent" (the `name` passed to define_deep_agent) in this workspace.
# Pass --name <name> here too if you deployed with a different --name in §2.6.
!cd "{project_root}/agents/deep_agent" && uv run mda delete --yes